## Evaluation - Part 2

In this notebook, we focus our efforts in evaluating how to best link to occupations. In the previous one, we found that linking titles to embeddings corresponding to a combination of all fields yielded better results than linking sentences describing the job to either combination of preferred label, secondary labels and description.

The second step includes the following questions:

1. If we allow multiple embeddings to correspond to a single node in the ESCO database, do we get a better recall?
2. What is the lowest value of k for which we get a recall of 1 (or, for lack of a better result, a sufficiently high value?)

We will define some preliminary functions and start with the first question:

In [1]:
# 1. Loading the test dataset for occupations using the Huggingface library
from huggingface_hub import hf_hub_download
import pandas as pd
from tqdm import tqdm
import os 
from dotenv import load_dotenv
from vertexai.language_models import TextEmbeddingModel

tqdm.pandas()

load_dotenv()

HF_TOKEN = os.environ["HF_ACCESS_TOKEN"]
OCCUPATION_REPO_ID = "tabiya/hahu_test"
OCCUPATION_FILENAME = "redacted_hahu_test_with_id.csv"
OCCUPATION_DATA_PATH = "https://raw.githubusercontent.com/tabiya-tech/taxonomy-model-application/main/data-sets/csv/tabiya-esco-v1.1.1/occupations.csv"


df_occupation_test = pd.read_csv(
    hf_hub_download(repo_id=OCCUPATION_REPO_ID, filename=OCCUPATION_FILENAME, repo_type="dataset", token=HF_TOKEN)
)
df_occupation_database = pd.read_csv(OCCUPATION_DATA_PATH)

model = TextEmbeddingModel.from_pretrained("textembedding-gecko@003")

/Users/francescopreta/miniconda3/envs/backend/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from typing import Any, Dict, List, Optional, Tuple

def precision_at_k(prediction: List[List[str]], true: List[List[str]], k: Optional[int] = None):
    """Calculates the average precision at k considering
    for each prediction the number of correct retrieved nodes
    divided by the number of total retrieved nodes.

    Args:
        prediction (List[List[str]]): list of 
            predicted lists, each with the corresponding
            nodes.
        true (List[List[str]]): list of the multiple true nodes 
            for each sample in the dataset.
        k (Optional[int]): number of samples of the prediction to consider.
            When None considers all the elements of the list.

    Returns:
        float: average precision at k over all the test set.
    """
    assert len(prediction) == len(true)
    total_precision = 0
    for pred_list, true_val in zip(prediction, true):
        if k:
            pred_list = pred_list[:k]
            tot_samples = k
        else:
            tot_samples = len(pred_list)
        total_precision+=len(set(pred_list).intersection(set(true_val)))/tot_samples
    return total_precision/len(true)

def recall_at_k(prediction: List[List[str]], true: List[List[str]], k: Optional[int] = None):
    """Calculates the average recall at k considering
    for each prediction the number of correct retrieved nodes
    divided by the number of total correct nodes.

    Args:
        prediction (List[List[str]]): list of 
            predicted lists, each with the corresponding
            nodes.
        true (List[List[str]]): list of the multiple true nodes 
            for each sample in the dataset.
        k (Optional[int]): number of predicted samples to consider.
            When None considers all of them. Defaults to None.

    Returns:
        float: average recall at k over all the test set.
    """
    assert len(prediction) == len(true)
    total_recall = 0
    for pred_list, true_val in zip(prediction, true):
        if k:
            pred_list = pred_list[:k]
        total_recall+=len(set(pred_list).intersection(set(true_val)))/len(set(true_val))
    return total_recall/len(true)

def f_score(prec: float, rec: float) -> float:
    """Returns the f-score corresponding to
    a given precision and recall.

    Args:
        prec (float): provided precision
        rec (float): provided recall

    Returns:
        float: resulting f-score
    """
    return 2*prec*rec/(prec+rec)

def get_all_metrics(predictions: List[List[str]], true_values: List[List[str]], k: Optional[int]=None) -> Tuple[float,float,float]:
    """Get recall, precision and F-score for given results and
    true values.

    Args:
        predictions (List[List[str]]): list of predictions.
        true_values (List[List[str]]): list of true values.
        k (Optional[int]): number of predicted samples to consider.
            When None considers all of them. Defaults to None.

    Returns:
        Tuple[float,float,float]: recall, precision and F-score.
    """
    rec_at_k = recall_at_k(predictions, true_values, k)
    prec_at_k = precision_at_k(predictions, true_values, k)
    f_score_at_k = f_score(prec_at_k, rec_at_k)
    return rec_at_k, prec_at_k, f_score_at_k

In [3]:
# Write a function to embed strings in batches
from typing import List

def embed_strings_in_batch(list_of_queries: List[str], model: TextEmbeddingModel, batch_size: int = 100) -> List[List[float]]:
    """Uses a TextEmbeddingModel to embed a list of queries in batches.

    Args:
        list_of_queries (List[str]): list of queries to be embedded in batches.
        model (TextEmbeddingModel): embedding model.
        batch_size (int, optional): size of each batch which should be less than or equal to 250.
            Defaults to 100.

    Returns:
        List[List[float]]: List of embeddings corresponding to the queries.
    """
    assert batch_size<=250
    embedding_results = []
    num_samples = len(list_of_queries)
    for i in range(int(num_samples/batch_size)+1):
        batch = list_of_queries[i*batch_size:(i+1)*batch_size]
        embedding_results += model.get_embeddings(batch)
    assert len(embedding_results) == len(list_of_queries)
    return [embedding_result.values for embedding_result in embedding_results]

### 1. Does segmentation through fields improve recall on Occupation?

In our previous experiment we wanted to find the best method to link the occupations of the test set to the ESCO database. We allowed each ESCO node to have a single embedding that could only be a combination of the strings in its fields (preferred label, secondary labels and description). We found that for low values of k, the preferred label guaranteed a higher recall, while higher values of k a combination of all fields guaranteed a better outcome.

We now want to consider the possibility that each node can be represented by multiple embeddings. This is similar to the way in which documents can be segmented and embedded in information retrieval. We consider two approaches:

1. We embed each document with three embeddings: preferred label, secondary labels and description. 
2. We embed each document with more than three embeddings, including one embedding for each secondary label.

We first generate the results and then compare their recall to the results of the previous evaluation. We compare the results for both synthetic queries and titles.

Notice that in our retrieval function, we consider k to be the set of unique top esco codes within the first 100 entries, meaning that the more embeddings per node we load in the collection, the least amount of different nodes we will find.

In [4]:
import chromadb

client = chromadb.Client()

def create_collections(db_name: str, df_database: pd.DataFrame):
    """Creates multiple collections for each choice of db_name
    and corresponding documents and embeddings.

    Args:
        db_name (str): name of the database. 
            Either 'skills' or 'occupations'.
        df_database (pd.DataFrame): database containing the metadata.
    """
    collection = client.create_collection(name=db_name)
    collection.add(
        documents = list(df_database["text"]),
        metadatas = [{"CODE": row["CODE"]} for _, row in df_database.iterrows()],
        embeddings = list(df_database["embedding"]),
        ids = [f"id_{i}" for i in range(len(df_database))]
    )

In [5]:
# Generate a dataframe having three rows for each node including the possible fields
from tqdm import tqdm 

df_occupation_full_list = []
for _, row in tqdm(df_occupation_database.iterrows()):
    for field in ["PREFERREDLABEL", "ALTLABELS", "DESCRIPTION"]:
        df_occupation_full_list.append({"text": str(row[field]), "CODE":row["CODE"]})
df_occupation_full_df = pd.DataFrame(df_occupation_full_list)
df_occupation_full_df["embedding"] = embed_strings_in_batch(list(df_occupation_full_df["text"]), model)

df_occupation_test["CODE"] = df_occupation_test["esco_code"].apply(lambda x: [x])

create_collections("occupation_three_embeddings", df_occupation_full_df)

3007it [00:00, 47037.37it/s]


In [6]:
from tqdm import tqdm 

df_occupation_full_list = []
for _, row in tqdm(df_occupation_database.iterrows()):
    for field in ["PREFERREDLABEL", "DESCRIPTION"]:
        df_occupation_full_list.append({"text": row[field], "CODE":row["CODE"]})
    for sec_label in str(row["ALTLABELS"]).split("\n"):
        df_occupation_full_list.append({"text": sec_label, "CODE":row["CODE"]})
df_occupation_full_df = pd.DataFrame(df_occupation_full_list)
df_occupation_full_df["embedding"] = embed_strings_in_batch(list(df_occupation_full_df["text"]), model)

df_occupation_test["CODE"] = df_occupation_test["esco_code"].apply(lambda x: [x])

create_collections("occupation_multiple_embeddings", df_occupation_full_df)

3007it [00:00, 42666.97it/s]


In [7]:
def get_results_from_embeddings(embeddings: List[List[float]], collection: chromadb.Collection, n_results:int=100) -> Tuple[List[List[str]], List[List[str]]]:
    """Utility function to return results of embedding queries
    to a given collection.

    Args:
        embeddings (List[List[float]]): List of embeddings for queries.
        collection (chromadb.Collection): ChromaDB collection to query.
        n_results (int): number of results to retrieve from the collection

    Returns:
        Tuple[List[List[str]], List[List[str]]]: List of results, one for
            regular vector search, the other one for maximal marginal relevance
            search. Each element is a list of string corresponding to the
            search result for the embedding in the same position in the input list. 
    """
    vector_search_results = []
    for embedding in embeddings:
        # Find the top 100 documents and save them in vector_search_results
        documents_from_search = collection.query(query_embeddings=embedding, n_results=n_results, include=["metadatas", "embeddings"])
        vector_search_results.append([elem["CODE"] for elem in documents_from_search["metadatas"][0]])
    return vector_search_results


def run_eval(collection_name: str, df_test: pd.DataFrame, target_column: str = "CODE", embedding_column = "embedding") -> pd.DataFrame:
    """Returns the results of an evaluation on a list of collections

    Args:
        collection_name (str): name of the collection.
        df_test (pd.DataFrame): test dataframe, containing an embedding column
            and a test_target column.

    Returns:
        pd.DataFrame: dataframe with the result of the evaluation depending on the
            different hyperparameters.
    """
    eval_data = []
    # Fetch collection
    collection = client.get_collection(collection_name)
    # Initialize lists to save results
    vector_search_results = get_results_from_embeddings(list(df_test[embedding_column]), collection)
    # Evaluate accuracy at k for k=1, 3, 5, 10
    single_vs_results = [list(set(elem)) for elem in vector_search_results]
    vector_search_results_single = [sorted(elem, key=vs_elem.index) for elem, vs_elem in zip(single_vs_results, vector_search_results)]
    for k in [1, 3, 5, 10]:
        rec_at_k, prec_at_k, f_score_at_k = get_all_metrics(vector_search_results_single, list(df_test[target_column]), k)
        eval_data.append({"method": collection_name, "embedded field": "title" if embedding_column=="embedding_title" else "synthetic query", "k":k, "recall": round(rec_at_k, 4), "precision": round(prec_at_k,4), "f-score": round(f_score_at_k,4)})
# Save the results in a dataframe
    eval_df = pd.DataFrame(eval_data)
    return eval_df

In [8]:
# Embed the titles and synthetic queries in the test set

df_occupation_test["embedding"] = embed_strings_in_batch(list(df_occupation_test["synthetic_query"]), model)
df_occupation_test["embedding_title"] = embed_strings_in_batch(list(df_occupation_test["title"]), model)

In [10]:
# Run the full evaluation for titles and synthetic queries
eval_df = pd.DataFrame()
for collection_name in ["occupation_multiple_embeddings", "occupation_three_embeddings"]:
    eval_df = pd.concat([eval_df, run_eval(collection_name, df_occupation_test)])
    eval_df = pd.concat([eval_df, run_eval(collection_name, df_occupation_test, embedding_column="embedding_title")])
    


The results are as follows, also including those from the previous evaluation on occupation

| Method                          | k  | Recall | Precision | F-score | Input Type       |
|---------------------------------|----|--------|-----------|---------|------------------|
| occupation_multiple_embeddings | 10 | 0.7657 | 0.0766    | 0.1392  | title            |
| occupation_three_embeddings    | 10 | 0.7601 | 0.076     | 0.1382  | synthetic query  |
| occupation_three_embeddings    | 10 | 0.7546 | 0.0755    | 0.1372  | title            |
| ALL_OCCUPATIONS                 | 10 | 0.7491 | 0.0749    | 0.1362  | title            |
| occupation_multiple_embeddings | 10 | 0.7435 | 0.0744    | 0.1352  | synthetic query  |
| ALL_OCCUPATIONS                 | 10 | 0.738  | 0.0738    | 0.1342  | synthetic query |
| occupation_three_embeddings    | 5  | 0.6919 | 0.1384    | 0.2306  | title            |
| occupation_three_embeddings    | 5  | 0.6919 | 0.1384    | 0.2306  | synthetic query  |
| ALL_OCCUPATIONS                 | 5  | 0.6882 | 0.1376    | 0.2294  | title            |
| ALL_OCCUPATIONS                 | 5  | 0.6605 | 0.1321    | 0.2202  | synthetic query  |
| occupation_multiple_embeddings | 5  | 0.6568 | 0.1314    | 0.2189  | title            |
| occupation_multiple_embeddings | 5  | 0.6494 | 0.1299    | 0.2165  | synthetic query  |
| occupation_three_embeddings    | 3  | 0.6421 | 0.214     | 0.321   | title            |
| ALL_OCCUPATIONS                 | 3  | 0.6162 | 0.2054    | 0.3081  | title            |
| occupation_three_embeddings    | 3  | 0.5959 | 0.1986    | 0.298   | synthetic query  |
| occupation_multiple_embeddings | 3  | 0.583  | 0.1943    | 0.2915  | title            |
| ALL_OCCUPATIONS                 | 3  | 0.5812 | 0.1937    | 0.2906  | synthetic query  |
| occupation_multiple_embeddings | 3  | 0.5646 | 0.1882    | 0.2823  | synthetic query  |
| occupation_three_embeddings    | 1  | 0.441  | 0.441     | 0.441   | title            |
| ALL_OCCUPATIONS                 | 1  | 0.4262 | 0.4262    | 0.4262  | title            |
| occupation_three_embeddings    | 1  | 0.3819 | 0.3819    | 0.3819  | synthetic query  |
| occupation_multiple_embeddings | 1  | 0.3579 | 0.3579    | 0.3579  | synthetic query  |
| occupation_multiple_embeddings | 1  | 0.3561 | 0.3561    | 0.3561  | title            |
| ALL_OCCUPATIONS                 | 1  | 0.3469 | 0.3469    | 0.3469  | synthetic query  |

We can notice the following trends:

- When using the **synthetic query**, embedding preferred labels, secondary labels and description separately is stably more effective than embedding them all together or than separating all the possible secondary labels. This makes sense, as we get some advantages over the multiple secondary labels embeddings by avoiding flooding the database with the same embedding for multiple labels. This is also stably better than the combination of all fields, probably because the embeddings are more focused and the more descriptive queries can be matched to the description directly, while those that are focused on the title can be matched to either preferred or secondary labels.
- When using the **title**, the method with three embeddings is more effective than all the others for low values of k. For higher values of k this doesn't hold anymore and the method with multiple embeddings has some gains as the titles are probably closer to each single preferred label. Since we don't care that most of the labels are not correct, this guarantees an advantage over the flooding of labels.

### 2. For which value of k do we obtain a sufficiently high recall value

Using the collections defined in the previous section, we now turn to the question of how many occupations should we iterate over if we want to find the correct job. The simplified assumption in this case is that every sample has exactly one correct ESCO node, while this might not be the case in theory. However, by getting a value when the true positive is only one per example gives us an important higher bound on how many occupations are needed to find the correct one in most cases.

In what follows, we would like to aim for a 100% recall. However, since this is not realistic in our experimental setting, we decide to aim for the highest recall such that the search doesn't repeatedly find any additional data as we increase k. The highest recall in this sense is the one that can stay constant for 100 values of k.

In [9]:
def get_highest_recall(
        collection: chromadb.Collection, 
        df_test: pd.DataFrame,
        embedding_column: str,
        target_column: str,
        n_results: int =100
        ) -> int:
    """Function to find the lowest value of k for which we get
    maximized recall (that is, the recall doesn't change for any
    larger k up to 100 higher).

    Args:
        collection (chromadb.Collection): collection from which
            we retrieve the occupation.
        df_test (pd.DataFrame): test dataframe with codes to retrieve.
        embedding_column (str): name of the embedding column in the test
            dataframe.
        target_column (str): name of the target column in the test dataframe.
        n_results (int): number of results to retrieve from the collection.

    Returns:
        Tuple[int, float]: lowest value of k giving the highest recall and
            highest value of recall
    """
    # Find vector search results
    vector_search_results = get_results_from_embeddings(list(df_test[embedding_column]), collection, n_results)
    # Find and order only one result per ESCO code
    single_vs_results = [list(set(elem)) for elem in vector_search_results]
    vector_search_results_single = [sorted(elem, key=vs_elem.index) for elem, vs_elem in zip(single_vs_results, vector_search_results)]
    # Define the lowest value of k and its recall
    k=1
    rec_at_k=0
    # Counter that will be reset everytime the recall improves
    improving_counter = 100
    while improving_counter>0 and rec_at_k<0.99:
        # Calculate recall at k
        temp_rec_at_k = recall_at_k(vector_search_results_single, list(df_test[target_column]), k)
        # If it's the same as before, reduce the counter
        if temp_rec_at_k<=rec_at_k:
            improving_counter-=1
        # Else reset the counter
        else:
            improving_counter = 100
        # Update recall and k
        rec_at_k = temp_rec_at_k
        k+=1
    if improving_counter==0:
        return k-100, rec_at_k
    return k, rec_at_k



We now evaluate the k and the recall for the two collections we have available

In [10]:
top_k, top_recall = get_highest_recall(client.get_collection("occupation_three_embeddings"), df_occupation_test, "embedding", "CODE")
print(f"For the collection with three embeddings per ESCO node, we get a maximum recall of {round(top_recall, 2)} at k={top_k}")
top_k, top_recall = get_highest_recall(client.get_collection("occupation_multiple_embeddings"), df_occupation_test, "embedding", "CODE")
print(f"For the collection with more than three embeddings per ESCO node, we get a maximum recall of {round(top_recall, 2)} at k={top_k}")


For the collection with three embeddings per ESCO node, we get a maximum recall of 0.9 at k=82
For the collection with more than three embeddings per ESCO node, we get a maximum recall of 0.88 at k=42


Interestingly enough, there is a trade-off between using more embeddings (thus having more chances of finding the right node) and flooding the number of entries for a single node. In our case, we get higher maximum recalls using only three embeddings, but we get the higher value earlier if we use more than three embeddings. This happens because we retrieve only a limited amount of embeddings at every iteration, so that the ranking is not done over the whole ESCO dataset. We can increase the number of embeddings we retrieve and observe how this trade-off is even more marked:

In [11]:
for n_results in [200, 500, 1000]:
    top_k, top_recall = get_highest_recall(client.get_collection("occupation_three_embeddings"), df_occupation_test, "embedding", "CODE", n_results)
    print(f"Within the first {n_results} results:")
    print(f"For the collection with three embeddings per ESCO node, we get a maximum recall of {round(top_recall, 2)} at k={top_k}")
    top_k, top_recall = get_highest_recall(client.get_collection("occupation_multiple_embeddings"), df_occupation_test, "embedding", "CODE", n_results)
    print(f"For the collection with more than three embeddings per ESCO node, we get a maximum recall of {round(top_recall, 2)} at k={top_k}")


Within the first 200 results:
For the collection with three embeddings per ESCO node, we get a maximum recall of 0.94 at k=145
For the collection with more than three embeddings per ESCO node, we get a maximum recall of 0.91 at k=79
Within the first 500 results:
For the collection with three embeddings per ESCO node, we get a maximum recall of 0.98 at k=379
For the collection with more than three embeddings per ESCO node, we get a maximum recall of 0.96 at k=165
Within the first 1000 results:
For the collection with three embeddings per ESCO node, we get a maximum recall of 0.99 at k=573
For the collection with more than three embeddings per ESCO node, we get a maximum recall of 0.98 at k=312


In [19]:
def find_code(true_code, list_of_codes):
    return true_code in list_of_codes


vector_search_results = get_results_from_embeddings(list(df_occupation_test["embedding"]), client.get_collection("occupation_three_embeddings"))
single_vs_results = [list(set(elem)) for elem in vector_search_results]
df_occupation_test["results"] = [sorted(elem, key=vs_elem.index) for elem, vs_elem in zip(single_vs_results, vector_search_results)]
df_occupation_residues = df_occupation_test[~df_occupation_test.apply(lambda x: find_code(x["CODE"][0], x["results"]), axis=1)]
df_esco_code_to_label = {row["CODE"]: row["PREFERREDLABEL"] for _, row in df_occupation_database.iterrows()}

df_occupation_residues["results"] = df_occupation_residues["results"].apply(lambda x:[df_esco_code_to_label[elem] for elem in x])
df_occupation_residues["CODE"] = df_occupation_residues["CODE"].apply(lambda x:df_esco_code_to_label[x[0]])

/var/folders/hy/zdgbwzss4dd84x7mp9nz9nj00000gn/T/ipykernel_26258/3901188977.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_occupation_residues["results"] = df_occupation_residues["results"].apply(lambda x:[df_esco_code_to_label[elem] for elem in x])
/var/folders/hy/zdgbwzss4dd84x7mp9nz9nj00000gn/T/ipykernel_26258/3901188977.py:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_occupation_residues["CODE"] = df_occupation_residues["CODE"].apply(lambda x:df_esco_code_to_label[x[0]])


In [20]:
df_occupation_residues[["title", "synthetic_query", "CODE", "results"]]

,title,synthetic_query,CODE,results
0,Paul Lacey International Programs (IP) Intern,I interned with AFSC as an International Progr...,monitoring and evaluation officer,"[immigration policy officer, international stu..."
10,Project Field Officer,I was a Project Field Officer for VIS in [REDA...,social work assistant,"[humanitarian advisor, human rights officer, p..."
17,Senior GBV Officer,"As a Senior GBV Officer, I oversaw the day-to-...",social care worker,"[sexual violence counsellor, humanitarian advi..."
21,Ethiopia Program Associate,I helped smallholder farmers in [REDACTED] imp...,programme manager,"[crop production worker, agricultural raw mate..."
23,Program Officer,"I supported program implementation, research, ...",programme manager,"[volunteer manager, activism officer, public a..."
32,Gender Based Violence Information Management S...,As the Gender Based Violence Information Manag...,community social worker,"[human rights officer, sexual violence counsel..."
62,Senior Research and Policy Specialist (Physica...,I was a Senior Research and Policy Specialist ...,university research assistant,"[education policy officer, educational researc..."
74,GA/ Medical Radiology Technologist,I worked as a Medical Radiology Technologist a...,assistant lecturer,"[radiographer, nuclear medicine radiographer, ..."
87,Skill Development Officer,As a Skill Development Officer at Amref Health...,business administration vocational teacher,"[youth worker, youth programme director, human..."
102,Woreda Project Coordinator,I coordinated projects with government offices...,project manager,"[international relations officer, political af..."
